# Task 2 : Training the model (Faster R-CNN and SSD)

This notebook performs the following steps:
1. Copies JPG images and XML annotations into a **working directory**
2. Removes incomplete JPG/XML pairs
3. Removes any image–annotation pair containing `<name>blank</name>`
4. Clean the Pascal VOC XML files.
5. Creates train and test split folders using `partition_dataset.py` 
6. Creates train and test tf records file using `generate_tfrecord.py`
7. Set path for the model and configure hyperparameters externally.
8. Execute the model using `python model_main_tf2.py`
9. Explanation of the hyperparameters used for training Faster R-CNN and SSD models.
10. Configration file references.

### STEP 1: SET Directory paths

In [ ]:
import os #!/usr/bin/env python3
import shutil # For file operations
import xml.etree.ElementTree as ET # For parsing XML files
import glob # For file pattern matching

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
BASE_PATH = r"C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/" # You can set this to your base path but this is my base path for tensorflow object detection API
OBJECT_DETECTION_PATH = fr"{BASE_PATH}/object_detection/"
SRC_IMG_DIR = fr"{BASE_PATH}/Tagging/Public/imageproject/dataset/" # Renom tag image directory
SRC_XML_DIR = fr"{BASE_PATH}/Tagging/Public/imageproject/label/detection/" # renom tag xml file directory

IMAGES_DIR = fr"{OBJECT_DETECTION_PATH}/images" 
os.makedirs(IMAGES_DIR, exist_ok=True)

print("Base dir        :", BASE_PATH)
print("Object detection dir:", OBJECT_DETECTION_PATH)
print("Source image dir:", SRC_IMG_DIR)
print("Source xml dir  :", SRC_XML_DIR)
print("Working dir     :", IMAGES_DIR)

Base dir        : C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/
Object detection dir: C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection/
Source image dir: C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//Tagging/Public/imageproject/dataset/
Source xml dir  : C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//Tagging/Public/imageproject/label/detection/
Working dir     : C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//images


### STEP 2: COPY JPGs and XMLs in images folder

In [2]:
jpg_count = 0  # counter for number of JPG files copied
for f in os.listdir(SRC_IMG_DIR):  # iterate over files in source image directory
    if f.lower().endswith('.jpg'):  # check file extension (case-insensitive)
        shutil.copy2(os.path.join(SRC_IMG_DIR, f), os.path.join(IMAGES_DIR, f))  # copy JPG into working images directory (preserves metadata)
        jpg_count += 1  # increment JPG counter

xml_count = 0  # counter for number of XML files copied
for f in os.listdir(SRC_XML_DIR):  # iterate over files in source XML directory
    if f.lower().endswith('.xml'):  # check for .xml extension
        shutil.copy2(os.path.join(SRC_XML_DIR, f), os.path.join(IMAGES_DIR, f))  # copy XML annotation to working images directory
        xml_count += 1  # increment XML counter

print(f"Copied {jpg_count} JPGs and {xml_count} XMLs into working directory")  # summary of copied files


Copied 1552 JPGs and 1552 XMLs into working directory


### STEP 3: Cleanup and remove incomplete and 'blank' pairs

This section scans the `images` working directory and performs two cleanup steps:

- **Remove incomplete pairs**: deletes files where either the `.jpg` or `.xml` is missing (keeps dataset consistent).
- **Remove `blank` labels**: parses each XML and removes both image and annotation if any object has `<name>blank</name>`.

The code builds sets of base filenames, identifies mismatches, deletes offending files, and prints a summary of how many pairs were removed.

In [ ]:
files = os.listdir(IMAGES_DIR)
            
# Create a set of base filenames (without extension) for JPG images
# Example: lion_001.jpg ->  lion_001
jpgs = {
    os.path.splitext(f)[0]
    for f in files
    if f.lower().endswith(".jpg")
}

# Create a set of base filenames (without extension) for XML annotation files
# Example: lion_001.xml -> lion_001
xmls = {
    os.path.splitext(f)[0]
    for f in files
    if f.lower().endswith(".xml")
}

# Counter to keep track of removed incomplete JPG/XML pairs
removed_incomplete = 0

# Counter to keep track of removed pairs containing 'blank' labels
removed_blank = 0

# Combine all unique basenames appearing in either JPG or XML files
# This helps detect missing pairs
all_basenames = jpgs.union(xmls)

# Loop over each unique base filename
for name in all_basenames:

    # Construct full path to corresponding JPG image
    jpg_path = os.path.join(IMAGES_DIR, name + ".jpg")

    # Construct full path to corresponding XML annotation
    xml_path = os.path.join(IMAGES_DIR, name + ".xml")

    # ---------- STEP 1: Remove incomplete pairs ----------
    # If either JPG or XML file is missing, the pair is incomplete
    if not os.path.exists(jpg_path) or not os.path.exists(xml_path):

        # Delete JPG if it exists
        if os.path.exists(jpg_path):
            os.remove(jpg_path)

        # Delete XML if it exists
        if os.path.exists(xml_path):
            os.remove(xml_path)

        # Increment counter for incomplete pairs
        removed_incomplete += 1

        # Skip further checks for this file
        continue

    # STEP 2: Remove blank-labeled pairs
    # Parse the XML annotation file
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # Check if any object in the annotation has label 'blank'
    has_blank = any(
        obj.find("name") is not None and
        obj.find("name").text.strip().lower() == "blank"
        for obj in root.findall("object")
    )

    # If a blank label is found, remove both image and annotation
    if has_blank:
        os.remove(jpg_path)
        os.remove(xml_path)

        # Increment counter for blank-labeled pairs
        removed_blank += 1

# Print summary of cleanup results
print(f"Removed incomplete pairs: {removed_incomplete}")
print(f"Removed blank-labeled pairs: {removed_blank}")

# Final confirmation message
print("Dataset is now ready for cleanup")


Removed incomplete pairs: 0
Removed blank-labeled pairs: 27
Dataset is now ready for cleanup


### STEP 4:Clean the Pascal VOC XML files

In [4]:
print("Cleaning XML files")

for xml_file in glob.glob(os.path.join(IMAGES_DIR, "*.xml")):
    with open(xml_file, "r") as f:
        content = f.read()

    # remove ALL whitespace including newlines & tabs
    content = content.replace("\n", "").replace("\t", "").replace(" ", "")

    with open(xml_file, "w") as f:
        f.write(content)

    print("Cleaned:", xml_file)

Cleaning XML files
Cleaned: C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//images\OryxGazella_CDB_S1_B06_R1_IMAG0225.xml
Cleaned: C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//images\OryxGazella_CDB_S1_B06_R1_IMAG0259.xml
Cleaned: C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//images\OryxGazella_CDB_S1_B06_R1_IMAG0268.xml
Cleaned: C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//images\OryxGazella_CDB_S1_B06_R1_IMAG0344.xml
Cleaned: C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//images\OryxGazella_CDB_S1_B06_R1_IMAG0345.xml
Cleaned: C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//images\OryxGazella_CDB_S1_B06_R1_IMAG0346.xml
Cleaned: C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//images\OryxGazella_CDB_S1_B06_R1_IMAG0347.xml
Cleaned: C:/Users/MSC1/Deskto

### STEP 5: Create train/test splits

This step runs `partition_dataset.py` to split the cleaned `images` folder into `images/train` and `images/test` (files are copied, not moved).

- **`-r` (ratio)**: fraction of data assigned to the *test* set. For example, `-r 0.1` → 10% test / 90% train.
- Files are copied so your original `images` directory remains intact.

In [5]:
!python {OBJECT_DETECTION_PATH}partition_dataset.py -x -i {IMAGES_DIR} -r 0.1

### STEP 6 : **Cleanup:** removes original `.jpg` and `.xml` files from the working `images` directory (these files were copied into `images/train` and `images/test` during splitting). This reduces duplication and saves disk space.


In [6]:
# ---------------- STEP 6: REMOVE ORIGINAL JPG AND XML FILES FROM IMAGES DIR ----------------
removed_files = 0
for f in os.listdir(IMAGES_DIR):
    if f.lower().endswith(('.jpg', '.xml')):
        os.remove(os.path.join(IMAGES_DIR, f))
        removed_files += 1

print(f"Removed {removed_files} original JPG and XML files from images directory")

Removed 3050 original JPG and XML files from images directory


### STEP 7: Generate TFRecords (train & test)

The commands below convert `images/train` and `images/test` into TensorFlow TFRecord files used for model training and evaluation.

- **Verify** `data/label_map.pbtxt` maps class names to numeric IDs correctly.
- **Check** that `images/train` and `images/test` contain the expected image/XML pairs before running.

**Commands executed below:**
```
!python generate_tfrecord.py -x images/train -l data/label_map.pbtxt -o data/train.record
!python generate_tfrecord.py -x images/test -l data/label_map.pbtxt -o data/test.record
```

After execution, confirm `data/train.record` and `data/test.record` exist and are non-empty.

In [7]:
# Create label_map.pbtxt
label_map_content = """item {
  id: 1
  name: "OryxGazella"
}
item {
  id: 2
  name: "StruthioCamelus"
}
item {
  id: 3
  name: "PhacochoerusAfricanus"
}"""

with open(os.path.join(OBJECT_DETECTION_PATH, 'data', 'label_map.pbtxt'), 'w') as f:
    f.write(label_map_content)

print("label_map.pbtxt created successfully.")

#Create the TF Record (Train)
!python {OBJECT_DETECTION_PATH}generate_tfrecord.py -x {IMAGES_DIR}/train -l {OBJECT_DETECTION_PATH}/data/label_map.pbtxt -o {OBJECT_DETECTION_PATH}/data/train.record

#Create the TF Record (Test)
!python {OBJECT_DETECTION_PATH}generate_tfrecord.py -x {IMAGES_DIR}/test -l {OBJECT_DETECTION_PATH}/data/label_map.pbtxt -o {OBJECT_DETECTION_PATH}/data/test.record

label_map.pbtxt created successfully.
Successfully created the TFRecord file: C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//data/train.record
Successfully created the TFRecord file: C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//data/test.record


### STEP 8: Start training & monitor with TensorBoard

To start training using `model_main_tf2.py` . Verify `MODEL_PATH` and `PIPELINE_CONFIG` above before running. The below code set paths for model and config file. I am using os.system to open a new command prompt window to run the training so that the current notebook remains responsive

In [9]:
# MODEL PATH AND PIPELINE CONFIG
MODEL_PATH = fr"{OBJECT_DETECTION_PATH}/training/TF2/training/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8"
# if you used a different model, change the above path accordingly. for ssd_resnet50_v1_fpn_640x640_coco17_tpu-8 comment out the above line and uncomment the below line
# MODEL_PATH = fr"{OBJECT_DETECTION_PATH}/training/TF2/training/ssd_resnet50_v1_fpn_640x640_coco17_tpu-8"

#I have trained both model using this notebook

PIPELINE_CONFIG = os.path.join(MODEL_PATH, "pipeline.config")

print("Model dir:", MODEL_PATH)
print("Pipeline config:", PIPELINE_CONFIG)
training_command = (
    f'python {OBJECT_DETECTION_PATH}model_main_tf2.py '
    f'--model_dir="{MODEL_PATH}" '
    f'--pipeline_config_path="{PIPELINE_CONFIG}" '
    f'--num_train_steps=6000 '
    f'--alsologtostderr'
)

print("Training command:", training_command)
os.system(f'start cmd /k {training_command}')

Model dir: C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//training/TF2/training/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8
Pipeline config: C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//training/TF2/training/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8\pipeline.config
Training command: python C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection/model_main_tf2.py --model_dir="C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//training/TF2/training/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8" --pipeline_config_path="C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1//object_detection//training/TF2/training/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8\pipeline.config" --num_train_steps=6000 --alsologtostderr


0

To monitor training metrics, run TensorBoard and open http://localhost:6006 in a browser:

---

# Hyperparameter Configuration and Explanation for Faster R-CNN ResNet-101

This section describes the key hyperparameters used for training the Faster R-CNN ResNet-101 model and explains their role in model performance.

---

## Number of Classes

```proto
num_classes: 3
```
In this coursework, three wildlife species were selected. The background class is handled internally by the TensorFlow Object Detection API.

---

## Image Resizer

```proto
fixed_shape_resizer {
  width: 1024
  height: 1024
}
```
**Purpose:**
- To ensure that images that have varying original resolutions can all be represented by the model in the same manner we use a fixed input size. Although the input size remains constant, we maintain the original aspect ratio of each image by padding these images automatically. This prevents the animals not only being stretched or squished, but also allows us to process a large number of images simultaneously during training.

- COCO assessment scores provide small, medium and large object detection, according to the size of the object with respect to the image. In this dataset, most of the image is occupied by animals as is typical of camera-trap images. Thus, on resizing, there are no items that belong to the small and medium size categories, and thus, such measures indicate -1.0.

- This is healthy and does not indicate that there is an error in training or assessment. It only indicates that the data set is dominated by big and noticeable animals in the shot.

This is how it is connects to TensorBoard plots. mAP (large) closely matches overall mAP .mAP (small / medium) remains −1.0 throughout training Recall (large) increases slightly, while other size categories remain undefined. This confirms that the model is learning effectively for the relevant object scale present in the dataset.

## Feature Extractor

```proto
type: "faster_rcnn_resnet101_keras"
batch_norm_trainable: true
```

The ResNet-101 backbone is used to extract high-level feature representations. Batch normalization layers remain trainable to adapt pretrained features to the target wildlife dataset.

---

## Anchor Generator

```proto
scales: [0.25, 0.5, 1.0, 2.0]
aspect_ratios: [0.5, 1.0, 2.0]
height_stride: 16
width_stride: 16
```
At every spatial location of the feature map, the model generates 12 anchor boxes by combining 4 scales with 3 aspect ratios. The feature-map stride of 16 defines the mapping between feature-map locations and the input image, meaning that each location corresponds to a 16×16 pixel region in the original image.

| Scale | Aspect Ratio | Approx Width × Height (pixels) | Interpretation |
|------|--------------|--------------------------------|----------------|
| 0.25 | 0.5 | ~45 × 90 | Small, tall anchor |
| 0.25 | 1.0 | ~64 × 64 | Small, square anchor |
| 0.25 | 2.0 | ~90 × 45 | Small, wide anchor |
| 0.5  | 0.5 | ~90 × 181 | Medium-small, tall anchor |
| 0.5  | 1.0 | ~128 × 128 | Medium-small, square anchor |
| 0.5  | 2.0 | ~181 × 90 | Medium-small, wide anchor |
| 1.0  | 0.5 | ~181 × 362 | Medium, tall anchor |
| 1.0  | 1.0 | ~256 × 256 | Medium, square (base anchor) |
| 1.0  | 2.0 | ~362 × 181 | Medium, wide anchor |
| 2.0  | 0.5 | ~362 × 724 | Large, tall anchor |
| 2.0  | 1.0 | ~512 × 512 | Large, square anchor |
| 2.0  | 2.0 | ~724 × 362 | Large, wide anchor |

---

## Region Proposal Network (First Stage)

### Non-Maximum Suppression

```proto
first_stage_nms_iou_threshold: 0.7
first_stage_max_proposals: 300
```

We established the threshold to 0.7 to make the initial stage as many possible objects as possible. The results are further refined by a second stage classifier. We consider 300 proposals to be sufficient and efficient, and we maintain final accuracy in the second step.

### Loss Weights

```proto
first_stage_localization_loss_weight: 2.0
first_stage_objectness_loss_weight: 1.0
```

The weights of losses explain the extent to which the individual components of the loss contribute to the loss in the total of RPN loss. A localization loss weight of 2.0 implies that errors in locating the precise box are doubly penalized as compared to errors in indicating whether a thing is a target or not. This challenges the RPN to study very precise boxes to fit object boundaries.

Conversely, an objectness loss weight of 1.0 provides a normal penalty to determining whether a region is a foreground or a background. It allows the model to reliably isolate objects in a background without stealing the entire training.

---

## Second Stage (Classification and Refinement)

```proto
score_threshold: 0.8
iou_threshold: 0.5
```

In the second step we narrow the results of the detection by the degree of certainty of the model and the extent of overlap of the boxes hence we obtain clear and trustworthy final results. The score threshold of 0.8 indicates that we only retain the detections whose probability of the model that we predict is at least 80 percent. This reduces false positives and makes the findings more accurate. We subsequently select using class-wise Non-Maximum Suppression (NMS) with an IoU threshold of 0.5. This implies that when two boxes have a greater value than half, we will only retain the box with the highest value and eliminate the others. These thresholds mean that the final decision is very strict and thus the final decision is reported only on very confident and non duplicate detections.

---

## Training Configuration

### Batch Size

```proto
batch_size: 2
```

The number of images that we train on once is called Batch size. The reason is that with a Faster R-CNN we require a large amount of memory, particularly with a ResNet-101 backbone and 1024 x 1024 images, which is why we set the batch size at 2. Larger batches would use more memory in the GPU and may lead to out-of-memory errors or reduce the stability of training.

### Optimizer

```proto
momentum_optimizer_value: 0.9
```

The momentum setting is used to control the extent to which the previous update is retained in developing the new update in the course of training. A momentum of 0.9 implies that we retain 90% of the last update and letting 10 percent of it come through the new gradient. It also smooths out updates, particularly in small batch sizes, such as when the model is large, such as Faster R-CNN with a ResNet -101 backbone.

---

## Learning Rate Schedule

```proto
cosine_decay_learning_rate {
  learning_rate_base: .005
  total_steps: 12000
  warmup_learning_rate: .00005
  warmup_steps: 2000
}
```

A cosine decay learning rate schedule with a warm-up phase was used to stabilise early training and ensure smooth convergence. The warm-up phase gradually increases the learning rate from 0.00005 to 0.005 over the first 2000 training steps, reducing the risk of unstable gradient updates. After warm-up, the learning rate decays smoothly following a cosine function until the final training step.

Training was conducted in two stages. Initially, the model was trained for 6000 steps, after which performance was evaluated on the validation set. To assess whether further optimisation would lead to meaningful improvements, training was then resumed for an additional 6000 steps (for a total of 12000 steps) using the same learning rate schedule and configuration.

Evaluation results showed only marginal changes in performance, with no consistent improvement in the primary COCO mAP metrics. This indicates that the model had largely converged by 6000 steps, and further training did not improve generalisation. Consequently, the checkpoint at 6000 steps was selected as the final model for all subsequent experiments and analysis.

---
```proto
fine_tune_checkpoint_version: V2
fine_tune_checkpoint: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/training/TF2/pre-trained-models/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8/checkpoint/ckpt-0"
fine_tune_checkpoint_type: "detection"
```

- `fine_tune_checkpoint_version`: V2 indicates that the checkpoint format follows version 2 of TensorFlow's checkpoint structure, which is required for compatibility with newer models. 
- `fine_tune_checkpoint` points to the exact file path ("C:/Users/MSC1/Desktop/.../ckpt-0") of the pre-trained model's weights, taken from a Faster R-CNN ResNet-101 model that was originally trained on the COCO dataset. 
- `fine_tune_checkpoint_type`: "detection" tells the training pipeline that the loaded checkpoint is from a full object detection model, so all relevant weights for bounding box prediction and classification will be restored and further fine-tuned on the new dataset instead of starting training from scratch. The model architecture rebuilds according to the numclasses value specified in  pipeline.config file which is 3 in my case, plus 1 internal background class, resulting in 4 output logits). The classification head (the final layers responsible for predicting class scores in the Region of Interest heads) has a different shape due to the reduced number of classes. Since the classification head shapes do not match, those weights are not loaded from the pre-trained checkpoint and are instead randomly initialized using the model's default initializer. The rest of the model including the backbone feature extractor (ResNet-101),  Region Proposal Network, and box regression head has compatible shapes, so those weights are successfully restored from the pre-trained checkpoint. This is the standard and intended behavior for transfer learning in object detection when the number of classes changes, allowing effective fine-tuning while adapting the head to your new classes.

---

## Data Augmentation

```proto
random_horizontal_flip
random_adjust_hue
random_adjust_contrast
random_adjust_saturation
random_square_crop_by_scale
```

When the model is training, we apply data-augmentation techniques to allow it to learn how to cope with the type of changes it is going to observe in real photos of wildlife camera-traps. We randomly invert images and this provides the model with various viewpoints and orientations. We also randomize color, contrast, and saturation; this is to make it appear as though lighting, time of day, weather, and camera sensor variations. Cropping a square area of the image randomly and at varying scale assists the model to be resilient to items that happen in various sizes and locations, compelling the model to study characteristics that are not scale-dependent. Our additional data augmentation consisted of the images we labelled since the TensorFlow Object Detection API has provided such techniques and they are automatically added in the training process. Through the setting of the augmentations in the configuration of the model, the same changes are enforced every time we train, and the original annotated dataset is not changed.

---

## Hardware Settings

```proto
use_bfloat16: false
```

The `bfloat16` precision mode is disabled because training was performed on a GPU-based system. This setting is only supported on TPUs.

---

## Training Input Reader

```proto
train_input_reader: {
  label_map_path: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/data/label_map.pbtxt"
  tf_record_input_reader {
    input_path: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/data/train.record"
  }
}
```

This section tells the model where the training data is located.The label_map_path defines the class names and their IDs used during training.The train.record file is a TFRecord file that contains the training images and their bounding box annotations Using TFRecord format improves training speed and efficiency, which is important for large models like Faster R-CNN.

---

## Evaluation Configuration

```proto
eval_config: {
  metrics_set: "coco_detection_metrics"
  use_moving_averages: false
  batch_size: 1
}
```

- `metrics_set: "coco_detection_metrics"`
Standard COCO metrics are used to measure detection performance, including mAP at different IoU thresholds.

- `use_moving_averages: false`
Evaluation uses the actual trained model weights instead of moving-average weights to keep results clear and consistent.

- `batch_size: 1`
One image is evaluated at a time, which is common for object detection and avoids memory issues.

```proto
eval_input_reader: {
  label_map_path: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/data/label_map.pbtxt"
  shuffle: false
  num_epochs: 1
  tf_record_input_reader {
    input_path: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/data/test.record"
  }
}
```
- The same label_map.pbtxt is used to keep class labels consistent.
- shuffle: false ensures the test images are evaluated in a fixed order.
- num_epochs: 1 means the test dataset is evaluated once.
- test.record contains unseen images used to measure how well the model generalises.

---
## Summary
The chosen settings mainly focuses on locations and reliably finding animals in wildlife pictures, all while working well with the available computer hardware.


In [10]:
os.system(f'start cmd /k tensorboard --logdir="{MODEL_PATH}" --port 6006')

0

## Faster RCNN Model Configurations for Reference

```proto
model {
  faster_rcnn {
    num_classes: 3
    image_resizer {
      fixed_shape_resizer {
        width: 1024
        height: 1024
      }
    }
    feature_extractor {
      type: "faster_rcnn_resnet101_keras"
      batch_norm_trainable: true
    }
    first_stage_anchor_generator {
      grid_anchor_generator {
        scales: [0.25, 0.5, 1.0, 2.0]
        aspect_ratios: [0.5, 1.0, 2.0]
        height_stride: 16
        width_stride: 16
      }
    }
    first_stage_box_predictor_conv_hyperparams {
      op: CONV
      regularizer {
        l2_regularizer { weight: 0.0 }
      }
      initializer {
        truncated_normal_initializer { stddev: 0.01 }
      }
    }
    first_stage_nms_score_threshold: 0.0
    first_stage_nms_iou_threshold: 0.7
    first_stage_max_proposals: 300
    first_stage_localization_loss_weight: 2.0
    first_stage_objectness_loss_weight: 1.0
    initial_crop_size: 14
    maxpool_kernel_size: 2
    maxpool_stride: 2

    second_stage_box_predictor {
      mask_rcnn_box_predictor {
        use_dropout: false
        dropout_keep_probability: 1.0
        fc_hyperparams {
          op: FC
          regularizer { l2_regularizer { weight: 0.0 } }
          initializer {
            variance_scaling_initializer {
              factor: 1.0
              uniform: true
              mode: FAN_AVG
            }
          }
        }
        share_box_across_classes: true
      }
    }

    second_stage_post_processing {
      batch_non_max_suppression {
        score_threshold: 0.8
        iou_threshold: 0.5
        max_detections_per_class: 100
        max_total_detections: 300
      }
      score_converter: SOFTMAX
    }

    second_stage_localization_loss_weight: 2.0
    second_stage_classification_loss_weight: 1.0
    use_static_shapes: true
    use_matmul_crop_and_resize: true
    clip_anchors_to_image: true
    use_static_balanced_label_sampler: true
    use_matmul_gather_in_matcher: true
  }
}

train_config: {
  batch_size: 2
  sync_replicas: false
  startup_delay_steps: 0
  replicas_to_aggregate: 1
  num_steps: 12000

  optimizer {
    momentum_optimizer {
      learning_rate {
        cosine_decay_learning_rate {
          learning_rate_base: .005
          total_steps: 12000
          warmup_learning_rate: .0005
          warmup_steps: 2000
        }
      }
      momentum_optimizer_value: 0.9
    }
    use_moving_average: false
  }

  fine_tune_checkpoint_version: V2
  fine_tune_checkpoint: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/training/TF2/pre-trained-models/faster_rcnn_resnet101_v1_1024x1024_coco17_tpu-8/checkpoint/ckpt-0"
  fine_tune_checkpoint_type: "detection"


  data_augmentation_options { random_horizontal_flip {} }
  data_augmentation_options { random_adjust_hue {} }
  data_augmentation_options { random_adjust_contrast {} }
  data_augmentation_options { random_adjust_saturation {} }

  data_augmentation_options {
    random_square_crop_by_scale {
      scale_min: 0.6
      scale_max: 1.3
    }
  }

  max_number_of_boxes: 100
  unpad_groundtruth_tensors: false
  use_bfloat16: false
}

train_input_reader: {
  label_map_path: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/data/label_map.pbtxt"
  tf_record_input_reader {
    input_path: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/data/train.record"
  }
}

eval_config: {
  metrics_set: "coco_detection_metrics"
  use_moving_averages: false
  batch_size: 1
}

eval_input_reader: {
  label_map_path: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/data/label_map.pbtxt"
  shuffle: false
  num_epochs: 1
  tf_record_input_reader {
    input_path: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/data/test.record"
  }
}
```

---

# Hyperparameter Configuration and Explanation for Single Shot Detector (SSD) – ResNet-50 FPN (640×640 Input Resolution)

This section explains the main hyperparameters used to train the SSD model and describes how each choice affects detection performance.

---

## Number of Classes

```proto
num_classes: 3
```

Three wildlife species are detected in this coursework.
The background class is handled internally by the TensorFlow Object Detection API and does not need to be specified explicitly.

---

## Image Resizer

```proto
fixed_shape_resizer {
  width: 640
  height: 640
}
```

**Purpose:**
All images are resized to a fixed resolution of 640×640 so that they can be processed efficiently by the SSD model. Using a fixed input size allows the model to perform detection in a single forward pass, which is a core advantage of SSD.

The original aspect ratio of images is preserved by padding, preventing distortion of animals. This ensures that objects are not stretched or squashed, which could negatively affect bounding box predictions.

Most animals in the dataset occupy a relatively large portion of the image. As a result, COCO metrics related to small objects may remain very low or undefined, while large-object mAP closely follows overall mAP. This behaviour is expected and confirms that the model is learning at the object scales present in the dataset.

---

## Feature Extractor

```proto
type: "ssd_resnet50_v1_fpn_keras"
```

The SSD model uses a ResNet-50 backbone combined with a Feature Pyramid Network (FPN).
ResNet-50 extracts strong semantic features, while the FPN allows detection at multiple scales by combining low-level and high-level features. This is especially useful for detecting animals appearing at different sizes and distances in camera-trap images.


```proto
regularizer {
  l2_regularizer {
   weight: 0.0004
  }
}

```

**Explanation**
L2 regularization adds a penalty to the loss function proportional to the square of the model weights. This discourages very large weights and forces the model to learn smoother and simpler decision boundaries. As a result, overfitting is reduced and generalization to unseen data improves.

**Value explanation:**

* Regularization weight (λ) = **0.0004**
* Larger λ → stronger penalty on large weights
* Smaller λ → weaker regularization, model fits training data more closely
* I am using a smaller value because my weigths will be in the range of +/-0.06

```proto
initializer {
    truncated_normal_initializer {
    mean: 0.0
    stddev: 0.03
  }
}
```

The truncated normal initializer assigns initial weights by sampling from a normal distribution centered at zero. Values that fall outside ±2 standard deviations are discarded to avoid extreme initial weights. This ensures stable and efficient training by starting the network with small, well-controlled weights.

**Value range:**

* Mean = 0.0
* Standard deviation = 0.03
* Allowed weight range ≈ **−0.06 to +0.06**

```proto
override_base_feature_extractor_hyperparams: true
```

The ResNet-50 backbone has its own default convolution settings. Setting this option to true ensures that the custom convolution hyperparameters defined in the configuration (such as L2 regularization, weight initializer, batch normalization, and activation function) are applied instead of the default ResNet settings. If this option were set to false, the backbone would continue using its original default parameters, and the custom regularizer, initializer, and batch-normalization settings specified in the configuration might be ignored.


```proto
fpn {
  min_level: 3
  max_level: 7
}
```
Although the input image size remains 640 × 640, the Feature Pyramid Network produces multiple feature maps at different resolutions. Anchor boxes are placed on each of these feature maps, allowing the model to detect objects of different sizes effectively.

| FPN Level | Approx Feature Map Size |
| --------- | ----------------------- |
| **P3**    | ~80 × 80                | 
| **P4**    | ~40 × 40                |
| **P5**    | ~20 × 20                |
| **P6**    | ~10 × 10                |
| **P7**    | ~5 × 5                  |


---

## Anchor Generator (SSD)

```proto
scales: [0.2, 0.35, 0.5, 0.65, 0.8, 0.95]
aspect_ratios: [1.0, 2.0, 0.5, 3.0, 0.333]

```

**Explanation:**

SSD generates default anchor boxes at multiple scales and aspect ratios across different feature maps.
The scale values are defined **relative to the input image size (640 × 640)**.

- Approximate anchor size = `scale × image_size`
- Width and height are then adjusted using the aspect ratio.
- Anchors larger than the image are allowed in SSD and are clipped internally; they improve recall for large or partially visible objects and do not harm training.

---

## SSD Anchor Size Approximation Table (640 × 640 Input)

| Scale | Aspect Ratio | Approx Width × Height (pixels) | Interpretation                 |
| ----- | ------------ | ------------------------------ | ------------------------------ |
| 0.20  | 1.0          | ~128 × 128                     | Small, square anchor           |
| 0.20  | 2.0          | ~181 × 90                      | Small, wide anchor             |
| 0.20  | 0.5          | ~90 × 181                      | Small, tall anchor             |
| 0.20  | 3.0          | ~222 × 74                      | Small, very wide anchor        |
| 0.20  | 0.333        | ~74 × 222                      | Small, very tall anchor        |
| 0.35  | 1.0          | ~224 × 224                     | Medium-small, square anchor    |
| 0.35  | 2.0          | ~317 × 158                     | Medium-small, wide anchor      |
| 0.35  | 0.5          | ~158 × 317                     | Medium-small, tall anchor      |
| 0.35  | 3.0          | ~387 × 129                     | Medium-small, very wide anchor |
| 0.35  | 0.333        | ~129 × 387                     | Medium-small, very tall anchor |
| 0.50  | 1.0          | ~320 × 320                     | Medium, square anchor          |
| 0.50  | 2.0          | ~453 × 226                     | Medium, wide anchor            |
| 0.50  | 0.5          | ~226 × 453                     | Medium, tall anchor            |
| 0.50  | 3.0          | ~554 × 185                     | Medium, very wide anchor       |
| 0.50  | 0.333        | ~185 × 554                     | Medium, very tall anchor       |
| 0.65  | 1.0          | ~416 × 416                     | Medium-large, square anchor    |
| 0.65  | 2.0          | ~588 × 294                     | Medium-large, wide anchor      |
| 0.65  | 0.5          | ~294 × 588                     | Medium-large, tall anchor      |
| 0.65  | 3.0          | ~719 × 240                     | Medium-large, very wide anchor |
| 0.65  | 0.333        | ~240 × 719                     | Medium-large, very tall anchor |
| 0.80  | 1.0          | ~512 × 512                     | Large, square anchor           |
| 0.80  | 2.0          | ~724 × 362                     | Large, wide anchor             |
| 0.80  | 0.5          | ~362 × 724                     | Large, tall anchor             |
| 0.80  | 3.0          | ~886 × 295                     | Large, very wide anchor        |
| 0.80  | 0.333        | ~295 × 886                     | Large, very tall anchor        |
| 0.95  | 1.0          | ~608 × 608                     | Very large, square anchor      |
| 0.95  | 2.0          | ~860 × 430                     | Very large, wide anchor        |
| 0.95  | 0.5          | ~430 × 860                     | Very large, tall anchor        |
| 0.95  | 3.0          | ~1054 × 351                    | Very large, very wide anchor   |
| 0.95  | 0.333        | ~351 × 1054                    | Very large, very tall anchor   |

---

## Key Difference vs Faster R-CNN

> Unlike Faster R-CNN, SSD applies these anchors **directly for classification and box regression in a single stage**, without a separate Region Proposal Network, making SSD faster but slightly less precise for small objects.

---

## Matching Strategy

```proto
matched_threshold: 0.5
unmatched_threshold: 0.4
```

* Anchors with IoU ≥ 0.5 are treated as positive examples.
* Anchors with IoU < 0.4 are treated as background.
* Anchors between these values are ignored.

This reduces ambiguity during training and helps the model learn clearer object–background separation.

---

## Loss Function

```proto
localization_loss_weight: 1.0
classification_loss_weight: 1.0
```

SSD optimizes two losses simultaneously:

* **Localization loss** penalizes inaccurate bounding box positions.
* **Classification loss** penalizes incorrect class predictions.

Equal weights ensure a balanced focus on both accurate box placement and correct species classification.

---

## Hard Example Mining

```proto
hard_example_miner {
  max_negatives_per_positive: 3
}
```

SSD generates a very large number of negative anchors (background).
Hard example mining keeps only the most difficult negative examples during training, preventing the model from being overwhelmed by easy background regions and improving learning efficiency.

---

## Training Configuration

### Batch Size

```proto
batch_size: 8
```

SSD is more memory-efficient than Faster R-CNN, allowing a larger batch size.
A batch size of 8 provides more stable gradient updates while remaining compatible with available GPU memory.

---

## Optimizer

```proto
momentum_optimizer_value: 0.9
```

Momentum helps smooth parameter updates by combining previous gradients with current gradients.
This is particularly helpful for SSD, where many anchors are optimized simultaneously, and training can otherwise be noisy.

---


## Learning Rate Schedule

```proto
cosine_decay_learning_rate {
  learning_rate_base: 0.005
  total_steps: 12000
  warmup_learning_rate: 0.0005
  warmup_steps: 2000
}
```

A cosine decay learning rate schedule with a warm-up phase was used to stabilise early training and ensure smooth convergence. The warm-up phase gradually increases the learning rate from 0.0005 to 0.005 over the first 2000 training steps, reducing the risk of unstable gradient updates. After warm-up, the learning rate decays smoothly following a cosine function until the final training step.

Training was conducted in two stages. Initially, the model was trained for 6000 steps, after which performance was evaluated on the validation set. To assess whether further optimisation would lead to meaningful improvements, training was then resumed for an additional 6000 steps (for a total of 12000 steps) using the same learning rate schedule and configuration.

Evaluation results showed only marginal changes in performance, with no consistent improvement in the primary COCO mAP metrics. This indicates that the model had largely converged by 6000 steps, and further training did not improve generalisation. Consequently, the checkpoint at 6000 steps was selected as the final model for all subsequent experiments and analysis.

---
```proto
fine_tune_checkpoint_version: V2
fine_tune_checkpoint: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/training/TF2/pre-trained-models/ssd_resnet50_v1_fpn_640x640_coco17_tpu-8/checkpoint/ckpt-0"
fine_tune_checkpoint_type: "detection"
```

- `fine_tune_checkpoint_version`: V2 indicates that the checkpoint format follows version 2 of TensorFlow's checkpoint structure, which is required for compatibility with newer models. 
- `fine_tune_checkpoint` points to the exact file path ("C:/Users/MSC1/Desktop/.../ckpt-0") of the pre-trained model's weights, taken from a ssd_resnet50_v1_fpn_640x640_coco17_tpu-8 model that was originally trained on the COCO dataset. 
- `fine_tune_checkpoint_type`: "detection" tells the training pipeline that the loaded checkpoint is from a full object detection model, so all relevant weights for bounding box prediction and classification will be restored and further fine-tuned on the new dataset instead of starting training from scratch. The model architecture rebuilds according to the numclasses value specified in  pipeline.config file which is 3 in my case, plus 1 internal background class, resulting in 4 output logits). The classification head (the final layers responsible for predicting class scores in the Region of Interest heads) has a different shape due to the reduced number of classes. Since the classification head shapes do not match, those weights are not loaded from the pre-trained checkpoint and are instead randomly initialized using the model's default initializer. The rest of the model has compatible shapes, so those weights are successfully restored from the pre-trained checkpoint. This is the standard and intended behavior for transfer learning in object detection when the number of classes changes, allowing effective fine-tuning while adapting the head to your new classes.

---

## Data Augmentation

```proto
random_horizontal_flip
random_adjust_brightness
random_adjust_contrast
random_adjust_saturation
random_crop_image
```

Data augmentation improves generalisation by exposing the model to variations commonly seen in wildlife imagery, such as changes in lighting, orientation, scale, and framing. These transformations help the model become more robust to real-world camera-trap conditions.

---

## Evaluation Configuration

```proto
metrics_set: "coco_detection_metrics"
batch_size: 1
```

Standard COCO detection metrics are used to evaluate performance.
Evaluation is performed one image at a time to avoid memory issues and ensure stable metric computation.

---

## Summary

The SSD model configuration focuses on **fast and efficient object detection**, making it suitable for wildlife datasets with limited hardware resources. By detecting objects in a single stage and using multi-scale anchors, SSD achieves a good balance between speed and accuracy, while data augmentation and transfer learning help improve generalisation on unseen images.

---

## SSD Model Configurations for Reference

```proto
model {
  ssd {
    num_classes: 3
    image_resizer {
      fixed_shape_resizer {
        height: 640
        width: 640
      }
    }
    feature_extractor {
      type: "ssd_resnet50_v1_fpn_keras"
      depth_multiplier: 1.0
      min_depth: 16
      conv_hyperparams {
        regularizer {
          l2_regularizer {
            weight: 0.0004
          }
        }
        initializer {
          truncated_normal_initializer {
            mean: 0.0
            stddev: 0.03
          }
        }
        activation: RELU_6
        batch_norm {
          decay: 0.99
          scale: true
          epsilon: 0.001
        }
      }
      override_base_feature_extractor_hyperparams: true
      fpn {
        min_level: 3
        max_level: 7
      }
    }
    box_coder {
      faster_rcnn_box_coder {
        y_scale: 10.0
        x_scale: 10.0
        height_scale: 5.0
        width_scale: 5.0
      }
    }
    matcher {
      argmax_matcher {
        matched_threshold: 0.5
        unmatched_threshold: 0.5
        ignore_thresholds: false
        negatives_lower_than_unmatched: true
        force_match_for_each_row: true
        use_matmul_gather: true
      }
    }
    similarity_calculator {
      iou_similarity {
      }
    }
    box_predictor {
      weight_shared_convolutional_box_predictor {
        conv_hyperparams {
          regularizer {
            l2_regularizer {
              weight: 0.0003
            }
          }
          initializer {
            random_normal_initializer {
              mean: 0.0
              stddev: 0.009
            }
          }
          activation: RELU_6
          batch_norm {
            decay: 0.996
            scale: true
            epsilon: 0.0010000000474974513
          }
        }
        depth: 256
        num_layers_before_predictor: 4
        kernel_size: 3
        class_prediction_bias_init: -4.60
      }
    }
    anchor_generator {
      multiscale_anchor_generator {
        min_level: 3
        max_level: 7
        anchor_scale: 4.0
        aspect_ratios: 1.0
        aspect_ratios: 2.0
        aspect_ratios: 0.5
        scales_per_octave: 2
      }
    }
    post_processing {
      batch_non_max_suppression {
        score_threshold: 0.5 
        iou_threshold: 0.5 
        max_detections_per_class: 100
        max_total_detections: 100
        use_static_shapes: false
      }
      score_converter: SIGMOID
    }
    normalize_loss_by_num_matches: true
    loss {
      localization_loss {
        weighted_smooth_l1 {
        }
      }
      classification_loss {
        weighted_sigmoid_focal {
          gamma: 2.0
          alpha: 0.25
        }
      }
      classification_weight: 1.0
      localization_weight: 1.0
    }
    encode_background_as_zeros: true
    normalize_loc_loss_by_codesize: true
    inplace_batchnorm_update: true
    freeze_batchnorm: false
  }
}
train_config {
  batch_size: 8
  data_augmentation_options {
    random_horizontal_flip {
    }
  }
  data_augmentation_options {
    random_crop_image {
      min_object_covered: 0.0
      min_aspect_ratio: 0.75
      max_aspect_ratio: 3.0
      min_area: 0.75
      max_area: 1.0
      overlap_thresh: 0.0
    }
  }
  sync_replicas: true
  replicas_to_aggregate: 1
  use_bfloat16: false
  optimizer {
    momentum_optimizer {
      learning_rate {
        cosine_decay_learning_rate {
          learning_rate_base: 0.005
          total_steps: 12000
          warmup_learning_rate: 0.0005
          warmup_steps: 2000
        }
      }
      momentum_optimizer_value: 0.9
    }
    use_moving_average: false
  }
  fine_tune_checkpoint: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/training/TF2/pre-trained-models/ssd_resnet50_v1_fpn_640x640_coco17_tpu-8/checkpoint/ckpt-0"
  num_steps: 12000
  startup_delay_steps: 0.0
  replicas_to_aggregate: 8
  max_number_of_boxes: 100
  unpad_groundtruth_tensors: false
  fine_tune_checkpoint_type: "detection"
  use_bfloat16: true
  fine_tune_checkpoint_version: V2
}
train_input_reader {
  label_map_path: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/data/label_map.pbtxt"
  tf_record_input_reader {
    input_path: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/data/train.record"
  }
}
eval_config {
  metrics_set: "coco_detection_metrics"
  use_moving_averages: false
}
eval_input_reader {
  label_map_path: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/data/label_map.pbtxt"
  shuffle: false
  num_epochs: 1
  tf_record_input_reader {
    input_path: "C:/Users/MSC1/Desktop/Tensorflow-Object-Detection-API/Base/v1/object_detection/data/test.record"
  }
}


```